# Lab 8 — OOP I: Core Concepts
**Difficulty: Intermediate | ~25 min | Requires Lab 3**

## Setup

In [1]:
# This lab uses only Python's built-in features, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!pip install --upgrade pip

## Step 1 — Define the Product class

In [2]:
# A class is a blueprint. The class name is written in CapWords by convention.
class Product:
    # __init__ runs automatically when we create a new object ("instance").
    def __init__(self, name, price, stock):
        # We store each value on `self`, the object being created.
        self.name = name    # public attribute: anyone can read or write it
        self.price = price  # public attribute
        self.stock = stock  # public attribute

    # A method is a function that belongs to the object (it takes `self`).
    def category(self):
        return "General merchandise"

    # Methods can also compute a value as a "question" we ask the object.
    def shipping_cost(self):
        return round(self.price * 0.08, 2)   # 8% of the price

    # __str__ controls what print(obj) shows -- a friendly one-line summary.
    def __str__(self):
        return f"{self.name} - ${self.price:.2f} (x{self.stock})"

## Step 2 — Create objects and use them

In [3]:
# Create two Product objects ("instances") from the same blueprint.
cap = Product("Cap", 14.99, 20)
hoodie = Product("Hoodie", 49.99, 8)

# print() calls __str__ automatically.
print(cap)
print(hoodie)

# We can read public attributes directly with dot notation.
print("Name:", cap.name)
print("Price:", cap.price)
print("Stock:", cap.stock)

# And we can call methods on an object.
print("Category:", cap.category())
print("Shipping for the cap:", cap.shipping_cost())
print("Shipping for the hoodie:", hoodie.shipping_cost())

Cap - $14.99 (x20)
Hoodie - $49.99 (x8)
Name: Cap
Price: 14.99
Stock: 20
Category: General merchandise
Shipping for the cap: 1.2
Shipping for the hoodie: 4.0


## Step 3 — Encapsulate the price with a property

In [4]:
# Improve Product so its price is protected: store it as _price (private by
# convention) and expose a property that validates every assigned value.
class Product:
    def __init__(self, name, price, stock):
        self.name = name
        # Single underscore = "private by convention": outside code should go
        # through the property below, not touch _price directly.
        self._price = price
        self.stock = stock

    # @property turns the method price() into an attribute read: product.price
    @property
    def price(self):
        return self._price

    # The setter runs when someone assigns: product.price = 19.99
    @price.setter
    def price(self, value):
        # Validation: reject any price below zero before it is stored.
        if value < 0:
            raise ValueError("Price cannot be negative")
        self._price = value

    def category(self):
        return "General merchandise"

    def shipping_cost(self):
        return round(self.price * 0.08, 2)

    def __str__(self):
        return f"{self.name} - ${self.price:.2f} (x{self.stock})"

# Build a product with the improved class.
cap = Product("Cap", 14.99, 20)

# Reading goes through the property getter.
print("Price via property:", cap.price)

# Assigning runs the setter, which accepts a valid value...
cap.price = 19.99
print("Price after update:", cap.price)

# ...and rejects an invalid one before it can be stored.
try:
    cap.price = -5
except ValueError as error:
    print("Rejected:", error)

Price via property: 14.99
Price after update: 19.99
Rejected: Price cannot be negative


## Step 4 — Name mangling with `__`

In [5]:
# Name mangling: a double underscore makes an attribute hard to reach
# from outside the class.
class Password:
    def __init__(self, value):
        # Double underscore triggers "name mangling" inside Python.
        self.__value = value

    def reveal(self):
        return self.__value   # inside the class, the name still works

password = Password("s3cr3t")

# From inside the class (reveal) we can read __value normally.
print("Reveal:", password.reveal())

# Outside, the plain name no longer exists -- this raises AttributeError.
try:
    print(password.__value)
except AttributeError as error:
    print("Direct access fails:", error)

# Python actually renames it to _Password__value (class name prefixed).
print("Mangled name:", password._Password__value)

Reveal: s3cr3t
Direct access fails: 'Password' object has no attribute '__value'
Mangled name: s3cr3t


## Step 5 — Inherit Book from Product

In [6]:
# Inheritance: Book reuses everything from Product and adds its own data.
class Book(Product):
    def __init__(self, name, price, stock, author, pages):
        # super() calls the parent constructor to set the shared attributes.
        super().__init__(name, price, stock)
        self.author = author   # extra attributes only a Book has
        self.pages = pages

    # Override: a Book uses a cheaper 5% shipping rate instead of 8%.
    def shipping_cost(self):
        return round(self.price * 0.05, 2)

    # Override that EXTENDS the parent: reuse the parent string, then add to it.
    def __str__(self):
        return super().__str__() + f" | by {self.author} ({self.pages}p)"

    # Override that EXTENDS the parent's category label.
    def category(self):
        return super().category() + " -> Books"

# Create a Book instance.
book = Book("Python Crash Course", 44.99, 15, "Eric Matthes", 544)
print(book)
print("isinstance(book, Product):", isinstance(book, Product))
print("isinstance(book, Book):", isinstance(book, Book))

Python Crash Course - $44.99 (x15) | by Eric Matthes (544p)
isinstance(book, Product): True
isinstance(book, Book): True


## Step 6 — Inherit Electronics from Product

In [7]:
# Electronics likewise specialises Product with brand and warranty data.
class Electronics(Product):
    def __init__(self, name, price, stock, brand, warranty_months):
        super().__init__(name, price, stock)
        self.brand = brand
        self.warranty_months = warranty_months

    # Override: electronics ship at 12% because they are heavier.
    def shipping_cost(self):
        return round(self.price * 0.12, 2)

    def __str__(self):
        return super().__str__() + f" | {self.brand} ({self.warranty_months}mo warranty)"

    def category(self):
        return super().category() + " -> Electronics"

# Create an Electronics instance.
phone = Electronics("Wireless Mouse", 29.99, 30, "LogiTech", 24)
print(phone)
print("isinstance(phone, Product):", isinstance(phone, Product))
print("isinstance(phone, Electronics):", isinstance(phone, Electronics))

Wireless Mouse - $29.99 (x30) | LogiTech (24mo warranty)
isinstance(phone, Product): True
isinstance(phone, Electronics): True


## Step 7 — Method overriding with super() in depth

In [8]:
# The parent's version, then each child's version of the same method.
print("Parent category:", Product("Generic", 5.0, 1).category())
print("Book category:", book.category())
print("Electronics category:", phone.category())

Parent category: General merchandise
Book category: General merchandise -> Books
Electronics category: General merchandise -> Electronics


## Step 8 — Polymorphism

In [9]:
# Polymorphism: the same method name, shipping_cost(), but each object
# runs its own class's version.
for item in [book, phone, cap]:
    print(f"{item.name}: ${item.shipping_cost():.2f}")

Python Crash Course: $2.25
Wireless Mouse: $3.60
Cap: $1.60


## Step 9 — Inventory report

In [10]:
# Treat every object as a Product; the right method runs automatically.
inventory = [
    cap,
    book,
    phone,
    Book("Fluent Python", 59.99, 10, "Luciano Ramalho", 792),
    Electronics("Bluetooth Speaker", 39.99, 12, "Sony", 12),
]

print("=== INVENTORY REPORT ===")
total_shipping = 0
for item in inventory:
    shipping = item.shipping_cost()
    total_shipping += shipping
    print(item)
print(f"Total shipping for all items: ${total_shipping:.2f}")

=== INVENTORY REPORT ===
Cap - $19.99 (x20)
Python Crash Course - $44.99 (x15) | by Eric Matthes (544p)
Wireless Mouse - $29.99 (x30) | LogiTech (24mo warranty)
Fluent Python - $59.99 (x10) | by Luciano Ramalho (792p)
Bluetooth Speaker - $39.99 (x12) | Sony (12mo warranty)
Total shipping for all items: $15.25
